# Grid Navigation with A\* Search
### AI 680 / CS 666 · Kyle Allen Sherman

---

This notebook explains the logic of A* Search section by section.

**What the code does:**  
A robot navigates a 10×10 grid from the top-left corner `(0,0)` to the bottom-right corner `(9,9)`, avoiding obstacles. It uses **A\* search** — which is essentially UCS upgraded with a heuristic — to find the shortest path while expanding as few nodes as possible.

**How A\* relates to what you already know:**

| Algorithm | Frontier order | Optimal? | Informed? |
|---|---|---|---|
| BFS | Fewest edges first | Yes (unweighted) | No |
| UCS | Lowest `g(n)` first | Yes | No |
| **A\*** | Lowest `f(n) = g(n) + h(n)` first | Yes (if h is admissible) | **Yes** |

The only new ingredient is `h(n)` — a heuristic estimate of the remaining cost to the goal. A\* uses it to prioritize nodes that are both cheap to reach *and* close to the goal.

---
## Section 1 — Imports

Only two imports are needed here:
- `heapq` — same min-heap used in UCS. A\* still needs to always expand the lowest-cost node next.
- `matplotlib.pyplot` — for drawing the grid.

In [ ]:
import heapq
import matplotlib.pyplot as plt

---
## Section 2 — The Grid

```python
grid = [
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    ...
]
```

The environment is encoded as a **2D list of integers**:
- `0` = open cell (traversable)
- `1` = wall / obstacle (blocked)

A cell is addressed as `grid[row][col]`, where `(0,0)` is the **top-left**.

**Important:** The grid's row/column indexing (`grid[r][c]`) is *separate* from the pixel coordinates used later during plotting. The conversion happens in `plot_maze` and is covered in Section 5.

### Why a 2D list and not a set of obstacles?
A list makes the grid-boundary check (`0 <= r < len(grid)`) clean and natural. A `set` of wall coordinates would also work but adds a conversion step; the list representation directly mirrors how grids are typically stored in robotics/game contexts.

In [ ]:
grid = [
    [0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 0, 1, 1, 1, 0],
    [0, 1, 0, 0, 0, 0, 0, 0, 1, 0],
    [0, 1, 0, 1, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 1, 0],
    [0, 1, 1, 1, 0, 1, 1, 1, 0, 0],
    [0, 1, 0, 0, 0, 1, 0, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 0, 0, 1, 0],
    [0, 1, 1, 0, 1, 1, 1, 0, 1, 0],
    [0, 0, 0, 0, 1, 1, 1, 0, 0, 0]
]

start = (0, 0)
goal  = (9, 9)

# Quick sanity check
print(f"Grid size: {len(grid)} rows × {len(grid[0])} cols")
print(f"Start cell value: {grid[start[0]][start[1]]}  (should be 0)")
print(f"Goal cell value:  {grid[goal[0]][goal[1]]}   (should be 0)")

---
## Section 3 — Movement Directions and the Heuristic

### Movement directions

```python
directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
#              Right    Down    Left      Up
```

Each tuple is a `(Δrow, Δcol)` offset. Adding one of these to the current cell's `(row, col)` produces its neighbor. **Only 4-directional movement** is allowed — no diagonals. This means every move costs exactly 1, keeping the cost model simple.

---

### The heuristic: Manhattan distance

```python
def heuristic(a, b):
    return abs(a[0] - b[0]) + abs(a[1] - b[1])
```

**Manhattan distance** counts how many horizontal + vertical steps it would take to reach `b` from `a` if there were *no obstacles*. It's the natural heuristic for a grid that only allows 4-directional movement.

**Why this heuristic is admissible (and why that matters):**  
A heuristic is *admissible* if it **never overestimates** the true remaining cost. Manhattan distance never overestimates because the actual path must make at least as many moves as the straight-line Manhattan count (obstacles can only add detours, never remove them). An admissible heuristic guarantees A\* finds the optimal path.

**What would happen with a bad heuristic?**  
If `h(n)` overestimated (e.g., `return 999`), A\* might skip over the true optimal path. If `h(n) = 0` for all nodes, A\* degenerates into plain UCS — still correct, but slower.

In [ ]:
directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]  # Right, Down, Left, Up

def heuristic(a, b):
    """Manhattan distance: admissible heuristic for 4-directional grid movement."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

# Demonstration
print(f"h(start → goal)  = {heuristic(start, goal)}")
print(f"h((0,0) → (0,0)) = {heuristic((0,0), (0,0))}  (same cell = 0)")
print(f"h((3,3) → (5,7)) = {heuristic((3,3), (5,7))}")

---
## Section 4 — A\* Search

### The heap entry format

```python
heapq.heappush(open_set, (f_cost, counter, g_cost, position, path))
```

Each entry packs five values:

| Field | Meaning |
|---|---|
| `f_cost` | `g + h` — the priority key; min-heap always pops the smallest |
| `counter` | Tiebreaker — if two nodes have equal `f`, the one added first wins. Prevents Python from trying to compare tuples like `(row, col)` |
| `g_cost` | Actual cost from start to this node |
| `position` | `(row, col)` of the current node |
| `path` | Full path list from start to here |

### `g_costs` dictionary

```python
g_costs = {start: 0}
```

Tracks the **best known `g` value** for each cell seen so far. This prevents re-adding a cell to the heap when we've already found a cheaper path to it — a key efficiency mechanism.

### `closed_set`

```python
closed_set = set()
```

Records fully **expanded** nodes. Once a node is popped from the heap and added here, any future heap entries for that node are skipped (`if current in closed_set: continue`). Because A\* with an admissible heuristic always finds the optimal path to a node on first expansion, this is safe.

### The neighbor loop

```python
for dr, dc in directions:
    neighbor = (current[0] + dr, current[1] + dc)
    if (0 <= neighbor[0] < len(grid) and       # within row bounds
        0 <= neighbor[1] < len(grid[0]) and    # within col bounds
        grid[neighbor[0]][neighbor[1]] == 0 and  # not a wall
        neighbor not in closed_set):            # not already expanded
```

Four conditions must all be true before a neighbor is considered. Order matters slightly: the bounds check comes first to avoid an `IndexError` before the grid value is accessed.

```python
new_g = g_cost + 1   # uniform cost — every step costs 1
if neighbor not in g_costs or new_g < g_costs[neighbor]:
    g_costs[neighbor] = new_g
    f = new_g + heuristic(neighbor, goal)
    heapq.heappush(open_set, (f, counter, new_g, neighbor, path + [neighbor]))
```

Only push the neighbor if this path to it is strictly better than any previously known path. `f = g + h` is computed fresh here with the neighbor's actual position, not the current node's.

In [ ]:
def a_star_search(grid, start, goal):
    """A* search: finds the shortest path using g(n) + h(n) as priority."""
    open_set = []
    counter  = 0   # tiebreaker — incremented on every push
    heapq.heappush(open_set, (heuristic(start, goal), counter, 0, start, [start]))
    # entry format: (f_cost, tiebreaker, g_cost, position, path)

    g_costs        = {start: 0}   # best known g-value per cell
    closed_set     = set()         # fully expanded nodes
    nodes_expanded = 0

    while open_set:
        f_cost, _, g_cost, current, path = heapq.heappop(open_set)

        if current in closed_set:   # stale entry — already expanded optimally
            continue

        closed_set.add(current)
        nodes_expanded += 1

        if current == goal:
            return path, g_cost, nodes_expanded

        for dr, dc in directions:
            neighbor = (current[0] + dr, current[1] + dc)
            if (0 <= neighbor[0] < len(grid) and
                0 <= neighbor[1] < len(grid[0]) and
                grid[neighbor[0]][neighbor[1]] == 0 and
                neighbor not in closed_set):

                new_g = g_cost + 1   # uniform move cost

                if neighbor not in g_costs or new_g < g_costs[neighbor]:
                    g_costs[neighbor] = new_g
                    f = new_g + heuristic(neighbor, goal)
                    counter += 1
                    heapq.heappush(open_set, (f, counter, new_g, neighbor, path + [neighbor]))

    return None, 0, nodes_expanded

# Run it
path, cost, nodes_expanded = a_star_search(grid, start, goal)

if path:
    print(f"Path found: {len(path)} steps, cost {cost}, {nodes_expanded} nodes expanded")
    print("Steps:", path)
else:
    print("No path found.")

---
## Section 5 — Visualisation

### The coordinate flip

This is the most confusing part of the plotting code. The grid uses `(row, col)` where `row=0` is the **top**, but matplotlib's y-axis goes **upward** (y=0 at the bottom). To flip row 0 to the top of the plot:

```python
y_pixel = rows - r - 1
```

So row 0 maps to `y = 9`, row 9 maps to `y = 0`. The x-axis uses column directly — columns already increase left-to-right, so no flip is needed.

### Drawing cells

```python
ax.add_patch(plt.Rectangle((c, rows - r - 1), 0.97, 0.97, color='black'))  # wall
ax.add_patch(plt.Rectangle((c, rows - r - 1), 0.97, 0.97, color='lightgray'))  # open
```

`plt.Rectangle((x, y), width, height)` — the `(x, y)` is the **bottom-left corner** of the rectangle. Width/height of `0.97` (not `1.0`) leaves a thin white gap between cells, creating a grid-line effect without actually drawing lines.

### Drawing the path arrows

```python
x1, y1 = path[i][1] + 0.5,    rows - path[i][0] - 0.5
x2, y2 = path[i+1][1] + 0.5,  rows - path[i+1][0] - 0.5
ax.arrow(x1, y1, x2 - x1, y2 - y1, ...)
```

The `+ 0.5` offsets place the arrow **center of cell** rather than corner. `ax.arrow` takes `(x, y, dx, dy)` — a starting point and a *delta*, not an endpoint. The delta is computed as `x2 - x1` and `y2 - y1`.

In [ ]:
def plot_maze(grid, path, start, goal):
    """Visualize the grid and the A* solution path."""
    rows, cols = len(grid), len(grid[0])
    fig, ax = plt.subplots(figsize=(10, 10))

    # Draw every cell — row 0 maps to top of plot via (rows - r - 1)
    for r in range(rows):
        for c in range(cols):
            color = 'black' if grid[r][c] == 1 else 'lightgray'
            ax.add_patch(plt.Rectangle((c, rows - r - 1), 0.97, 0.97, color=color))

    # Draw path as red arrows between cell centers
    if path:
        for i in range(len(path) - 1):
            x1, y1 = path[i][1] + 0.5,     rows - path[i][0] - 0.5
            x2, y2 = path[i+1][1] + 0.5,   rows - path[i+1][0] - 0.5
            ax.arrow(x1, y1, x2 - x1, y2 - y1,
                     head_width=0.12, length_includes_head=True,
                     head_length=0.12, fc='red', ec='red')

    # Mark start (green) and goal (blue) as smaller centered squares
    ax.add_patch(plt.Rectangle((start[1] + 0.25, rows - start[0] - 1 + 0.25), 0.5, 0.5, color='green'))
    ax.add_patch(plt.Rectangle((goal[1]  + 0.25, rows - goal[0]  - 1 + 0.25), 0.5, 0.5, color='blue'))

    ax.set_xlim(0, cols)
    ax.set_ylim(0, rows)
    ax.set_xticks(range(cols + 1))
    ax.set_yticks(range(rows + 1))
    ax.grid(True, color='white', linewidth=0.5)
    ax.set_title("Robot Grid Navigation - A* Shortest Path\n(Green = Start, Blue = Goal, Red = Path)")
    plt.show()

plot_maze(grid, path, start, goal)

---
## Summary

### What makes A\* different from UCS

The only structural difference is in how `f` is computed when pushing to the heap:

| Algorithm | Priority key |
|---|---|
| UCS | `f = g(n)` |
| A\* | `f = g(n) + h(n)` |

Adding `h(n)` biases the search toward cells that are already close to the goal, so A\* doesn't waste time expanding nodes in the wrong direction. On this 10×10 grid, that can mean the difference between expanding most of the grid vs. a narrow corridor of nodes.

### Admissibility recap

- **Admissible heuristic:** `h(n) ≤ true remaining cost` — A\* is guaranteed optimal
- **Manhattan distance** is admissible for 4-directional grids because every move costs 1 and you can never do better than moving in a straight line
- If diagonals were allowed (8-directional), you'd switch to **Chebyshev distance** (`max(|Δr|, |Δc|)`)
- If move costs varied, you'd need a different heuristic scaled to the minimum edge cost

### `nodes_expanded` as a quality metric

The counter tracks how informed the search was. A better (tighter) heuristic → fewer nodes expanded → faster search. You could compare A\* against UCS on this grid by zeroing out the heuristic and observing the difference in `nodes_expanded`.

In [ ]:
# Bonus: compare A* vs UCS (h=0) node expansion count
def ucs_as_astar(grid, start, goal):
    """A* with h=0 — equivalent to UCS on a grid."""
    open_set = []
    counter  = 0
    heapq.heappush(open_set, (0, counter, 0, start, [start]))
    g_costs        = {start: 0}
    closed_set     = set()
    nodes_expanded = 0
    while open_set:
        f_cost, _, g_cost, current, path = heapq.heappop(open_set)
        if current in closed_set:
            continue
        closed_set.add(current)
        nodes_expanded += 1
        if current == goal:
            return path, g_cost, nodes_expanded
        for dr, dc in directions:
            neighbor = (current[0] + dr, current[1] + dc)
            if (0 <= neighbor[0] < len(grid) and
                0 <= neighbor[1] < len(grid[0]) and
                grid[neighbor[0]][neighbor[1]] == 0 and
                neighbor not in closed_set):
                new_g = g_cost + 1
                if neighbor not in g_costs or new_g < g_costs[neighbor]:
                    g_costs[neighbor] = new_g
                    counter += 1
                    heapq.heappush(open_set, (new_g, counter, new_g, neighbor, path + [neighbor]))
    return None, 0, nodes_expanded

_, _, ucs_expanded = ucs_as_astar(grid, start, goal)
_, _, astar_expanded = a_star_search(grid, start, goal)

print(f"UCS  (h=0) nodes expanded: {ucs_expanded}")
print(f"A*          nodes expanded: {astar_expanded}")
print(f"A* saved {ucs_expanded - astar_expanded} node expansions ({100*(ucs_expanded-astar_expanded)/ucs_expanded:.1f}% fewer)")